# 03 Task 1 Results And Analysis

Load the standardized Task 1 outputs from `runs/task1/` and generate summary tables plus analysis plots.


In [ ]:
from pathlib import Path
import json
import os
import sys

import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

ROOT


In [ ]:
SCENARIO_STYLE = {
    "real": {"color": "#2196F3", "marker": "o", "label": "Real only"},
    "real_aug": {"color": "#8E24AA", "marker": "D", "label": "Real + classical aug"},
    "synth": {"color": "#FF9800", "marker": "s", "label": "Synth only"},
    "both": {"color": "#4CAF50", "marker": "^", "label": "Real + Synth"},
}
SCENARIO_ORDER = ["real", "real_aug", "synth", "both"]


def group_by_scenario(results):
    grouped = {}
    for result in results:
        grouped.setdefault(result["scenario"], []).append(result)
    for runs in grouped.values():
        runs.sort(key=lambda item: item["n_per_class"])
    return {scenario: grouped[scenario] for scenario in SCENARIO_ORDER if scenario in grouped}


def time_value(result, time_field="pipeline_time_sec"):
    if time_field == "pipeline_time_sec":
        return result.get("pipeline_time_sec", result.get("train_time_sec", 0.0))
    if time_field == "classifier_train_time_sec":
        return result.get("classifier_train_time_sec", result.get("train_time_sec", 0.0))
    return result.get(time_field, result.get("train_time_sec", 0.0))


def print_accuracy_table(results):
    grouped = group_by_scenario(results)
    sizes = sorted({run["n_per_class"] for run in results})
    header = ["Size"] + [SCENARIO_STYLE[scenario]["label"] for scenario in grouped]
    print(" | ".join(f"{col:>16}" for col in header))
    print("-" * (19 * len(header)))
    for size in sizes:
        row = [f"{size:>16}"]
        for scenario, runs in grouped.items():
            match = next((run for run in runs if run["n_per_class"] == size), None)
            if match is None:
                row.append(f"{'N/A':>16}")
            else:
                row.append(f"{match['test_accuracy'] * 100:15.2f}%")
        print(" | ".join(row))


def summarize_pipeline(pipeline_summary):
    if not pipeline_summary:
        print("No pipeline summary found.")
        return
    print("\nPer-size pipeline summary:")
    for item in pipeline_summary:
        n = item["n_per_class"]
        gan = item.get("gan", {})
        synth = item.get("synth", {})
        print(
            f"n={n}: best_fid={gan.get('best_fid')} best_epoch={gan.get('best_epoch')} "
            f"gan_time={gan.get('train_time_sec')}s synth_time={synth.get('generate_time_sec')}s"
        )


def print_best_runs(results):
    grouped = group_by_scenario(results)
    print("\nBest run per scenario:")
    for scenario, runs in grouped.items():
        best = max(runs, key=lambda item: item["test_accuracy"])
        print(
            f"{SCENARIO_STYLE[scenario]['label']}: n={best['n_per_class']} "
            f"acc={best['test_accuracy'] * 100:.2f}% pipeline_time={time_value(best):.1f}s"
        )


def plot_results_notebook(results, out_dir, time_field="pipeline_time_sec"):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    grouped = group_by_scenario(results)

    fig, ax = plt.subplots(figsize=(8, 5))
    for scenario, runs in grouped.items():
        style = SCENARIO_STYLE[scenario]
        ax.plot(
            [run['n_per_class'] for run in runs],
            [run['test_accuracy'] * 100 for run in runs],
            marker=style['marker'],
            color=style['color'],
            label=style['label'],
            linewidth=2,
            markersize=8,
        )
    ax.set_xlabel("Training images per class")
    ax.set_ylabel("Test Accuracy (%)")
    ax.set_title("Classification Accuracy vs Training Data Size")
    ax.legend()
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(out_dir / "accuracy_vs_size.png", dpi=150)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(8, 5))
    for scenario, runs in grouped.items():
        style = SCENARIO_STYLE[scenario]
        ax.plot(
            [run['n_per_class'] for run in runs],
            [time_value(run, time_field=time_field) for run in runs],
            marker=style['marker'],
            color=style['color'],
            label=style['label'],
            linewidth=2,
            markersize=8,
        )
    ax.set_xlabel("Training images per class")
    ax.set_ylabel("Time (seconds)")
    ax.set_title("Runtime vs Training Data Size")
    ax.legend()
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(out_dir / "time_vs_size.png", dpi=150)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(9, 5))
    scenarios = [scenario for scenario in SCENARIO_ORDER if scenario in grouped]
    class_names = ["apple", "banana", "orange"]
    width = 0.18
    x_positions = list(range(len(class_names)))
    for offset, scenario in enumerate(scenarios):
        runs = grouped[scenario]
        best = max(runs, key=lambda item: item['test_accuracy'])
        scores = [best['per_class'][class_name]['f1'] for class_name in class_names]
        positions = [x + (offset - (len(scenarios) - 1) / 2) * width for x in x_positions]
        ax.bar(positions, scores, width=width, color=SCENARIO_STYLE[scenario]['color'], label=SCENARIO_STYLE[scenario]['label'])
    ax.set_xticks(x_positions)
    ax.set_xticklabels(class_names)
    ax.set_ylim(0, 1)
    ax.set_ylabel("F1 Score")
    ax.set_title("Per-Class F1 For The Best Run Per Scenario")
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    fig.tight_layout()
    fig.savefig(out_dir / "per_class_f1.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    return grouped


In [ ]:
# Edit these paths before running.
OUT_ROOT = ROOT / "runs" / "task1"
RESULTS_JSON = OUT_ROOT / "clf" / "all_results.json"
PIPELINE_SUMMARY_JSON = OUT_ROOT / "pipeline_summary.json"
PLOTS_DIR = OUT_ROOT / "clf" / "plots"

RESULTS_JSON, PIPELINE_SUMMARY_JSON


In [ ]:
results = []
pipeline_summary = []

if RESULTS_JSON.exists():
    results = json.loads(RESULTS_JSON.read_text())
    print(f"Loaded {len(results)} classifier results from {RESULTS_JSON}")
else:
    print(f"Missing results file: {RESULTS_JSON}")
    print("Run notebooks/02_generate_synthetic_data_and_classifier_experiments.ipynb first to generate Task 1 outputs.")

if PIPELINE_SUMMARY_JSON.exists():
    pipeline_summary = json.loads(PIPELINE_SUMMARY_JSON.read_text())
    print(f"Loaded pipeline summary from {PIPELINE_SUMMARY_JSON}")
elif results:
    print(f"Missing pipeline summary file: {PIPELINE_SUMMARY_JSON}")


In [ ]:
if results:
    print_accuracy_table(results)
    print_best_runs(results)
    summarize_pipeline(pipeline_summary)
else:
    print("No Task 1 outputs are available yet.")


In [ ]:
if results:
    grouped = plot_results_notebook(results, PLOTS_DIR, time_field="pipeline_time_sec")
    print(f"Saved plots to {PLOTS_DIR}")
    list(grouped.keys())
